# Primer taller: árboles de regresión y clasificación en Python

**Estadística II — Universidad Nacional de Colombia, Sede Medellín**  
**Período:** 2026-II · **Monitor:** Tomás Rodríguez Taborda

Este notebook es una plantilla para resolver la parte computacional del primer taller de árboles e interpretabilidad. Consulte [PrimerTaller.pdf](PrimerTaller.pdf) para los enunciados de la **parte teórica**. Desarrolle las demostraciones y justificaciones en su entrega.

Complete las celdas de código y las interpretaciones de los ocho ejercicios. Ejecute el notebook de arriba hacia abajo, conserve sus resultados y explique qué significan. Toda la parte computacional se realiza en **Python**.

## Preparación

Abra este notebook con un kernel de Python 3.11 o posterior. Si faltan dependencias, descomente y ejecute la siguiente línea desde la carpeta del taller y reinicie el kernel cuando termine. `requirements.txt` está en esta misma carpeta.

In [ ]:
# Instalación opcional; ejecutar únicamente si faltan dependencias.
# %pip install -r requirements.txt

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from ISLP import load_data
from sklearn.base import clone
from sklearn.dummy import DummyRegressor
from sklearn.linear_model import LinearRegression, LogisticRegression
from sklearn.metrics import accuracy_score, confusion_matrix, mean_squared_error
from sklearn.model_selection import KFold, cross_val_score
from sklearn.pipeline import make_pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.tree import DecisionTreeClassifier, DecisionTreeRegressor, plot_tree

SEMILLA = 2026

## Datos y partición

`Hitters`, disponible en `ISLP`, contiene el salario de 1987 y las estadísticas de desempeño de **322 jugadores** de las Grandes Ligas de béisbol. Use únicamente:

- **Salary:** salario anual en miles de dólares.
- **Years:** años en las Grandes Ligas.
- **Hits:** hits (batazos válidos) de la temporada de 1986.

Elimine los jugadores sin salario registrado: quedan **263**. La respuesta es el **logaritmo natural** del salario, $Y=\log(\mathrm{Salary})$. Tome las primeras **200 filas** como entrenamiento y las **63 restantes** como prueba, **sin aleatorizar** y conservando el orden original después de eliminar los salarios faltantes.

In [ ]:
hitters = (
    load_data("Hitters")
    .dropna(subset=["Salary"])
    .loc[:, ["Salary", "Years", "Hits"]]
    .reset_index(drop=True)
)
X = hitters[["Years", "Hits"]]
y = np.log(hitters["Salary"]).rename("log(Salary)")

X_ent, X_pru = X.iloc[:200].copy(), X.iloc[200:].copy()
y_ent, y_pru = y.iloc[:200].copy(), y.iloc[200:].copy()

assert len(hitters) == 263
assert len(X_ent) == len(y_ent) == 200
assert len(X_pru) == len(y_pru) == 63
assert (hitters["Salary"] > 0).all()

hitters[["Salary", "Years", "Hits"]].head()

## 1. Respuesta y predicción sin predictoras

Grafique $\log(\mathrm{Salary})$ frente a **Years** y frente a **Hits** en el entrenamiento, y **explique** por qué se modela el logaritmo del salario. Obtenga la predicción que se haría sin usar las predictoras y su RSS de entrenamiento.

In [ ]:
# Grafique la respuesta frente a cada predictora usando el entrenamiento.
# Calcule la predicción constante y su RSS de entrenamiento.

**Interpretación y respuesta:**

_Escriba aquí su análisis y justificación._

## 2. Búsqueda de la primera división

**Programe una función** que, para una predictora numérica, evalúe todos los cortes admisibles (puntos medios entre valores consecutivos distintos que dejan al menos **10 jugadores a cada lado**) y devuelva el RSS izquierdo, el derecho y el total de cada uno. Aplíquela a **Years** y a **Hits**, grafique el RSS total frente al punto de corte y reporte el número de cortes admisibles, el mejor corte de cada predictora y la primera división del árbol.

In [ ]:
# Programe aquí la función de evaluación de cortes.
# Aplíquela a Years y Hits; compare los criterios y grafique los resultados.

**Interpretación y respuesta:**

_Escriba aquí su análisis y justificación._

## 3. Árbol de regresión y regiones

Ajuste con `DecisionTreeRegressor` un árbol de regresión con profundidad máxima **2**, al menos **20 jugadores** para intentar una división y **10 por hoja**. Dibújelo con `plot_tree`, reporte la regla, el número de jugadores y la predicción de cada hoja, y represente la partición del plano $(\mathrm{Years},\mathrm{Hits})$. **Describa las cuatro regiones**. ¿Por qué el corte en **Hits** es distinto en cada rama?

In [ ]:
# Ajuste DecisionTreeRegressor con los parámetros indicados.
# Dibuje el árbol y la partición del plano; reporte la información de cada hoja.

**Interpretación y respuesta:**

_Escriba aquí su análisis y justificación._

## 4. Predicciones y explicación de las reglas

Para tres jugadores nuevos con $(\mathrm{Years},\mathrm{Hits})$ iguales a $(3,150)$, $(7,90)$ y $(12,160)$, siga las reglas del árbol hasta su hoja, obtenga la predicción y verifíquela con la función de predicción. Exprésela en miles de dólares mediante $e^{\widehat{y}}$ y **explique** por qué este valor no es el salario promedio de los jugadores de la hoja.

In [ ]:
# Cree un DataFrame con los tres jugadores nuevos y las columnas Years y Hits.
# Siga las reglas de cada ruta, compruebe con predict y transforme con np.exp.

**Interpretación y respuesta:**

_Escriba aquí su análisis y justificación._

## 5. Comparación de modelos de regresión

Compare el RSS, el MSE y el RMSE de entrenamiento y de prueba de la media de entrenamiento, del árbol del punto 3 y de una regresión lineal múltiple con **Years** y **Hits**. ¿Qué modelo predice mejor a los jugadores de prueba? **Relacione** el resultado con los gráficos del punto 1.

In [ ]:
# Ajuste los tres modelos con el entrenamiento.
# Construya una tabla de RSS, MSE y RMSE para entrenamiento y prueba.

**Interpretación y respuesta:**

_Escriba aquí su análisis y justificación._

## 6. Profundidad y validación cruzada

Para profundidades máximas de **1 a 6**, obtenga el MSE de entrenamiento y el de validación cruzada de **10 particiones**, calculado **solo con el entrenamiento**. Elija la profundidad que **minimiza el MSE de validación cruzada**; en caso de empate, prefiera la **menor**. Después calcule el MSE de prueba de cada profundidad y grafique las tres curvas, **sin cambiar la elección anterior**.

**Justifique** por qué la profundidad se elige con validación cruzada y no con el conjunto de prueba. Relacione la curva de entrenamiento con el punto **1(b) de la parte teórica**.

In [ ]:
# Calcule MSE de entrenamiento y validación cruzada para profundidades 1 a 6.
# Use KFold(n_splits=10, shuffle=False) y cambie el signo del MSE negativo.
# Elija el menor MSE de validación cruzada; desempate por menor profundidad.
# Después calcule el MSE de prueba y grafique las tres curvas.
# Mantenga la profundidad elegida antes de consultar la curva de prueba.

**Interpretación y respuesta:**

_Escriba aquí su análisis y justificación._

## 7. Árbol de clasificación y probabilidades

Defina la clase **Alto** si $\mathrm{Salary}\geq 500$ y **Bajo** en otro caso, y reporte su proporción en el entrenamiento y en la prueba. Adapte la función del punto 2 al **Gini ponderado** para hallar la primera división, y ajuste con `DecisionTreeClassifier` un árbol de clasificación con el índice de Gini y profundidad máxima **2** y los mismos tamaños mínimos del punto 3.

Reporte las proporciones y la clase predicha de cada hoja, identifique una división cuyas dos hojas predicen la misma clase y **relaciónela** con el punto **3(c) de la parte teórica**. Para los jugadores del punto 4, obtenga $\widehat{P}(\mathrm{Alto})$ con `predict_proba` y la clase predicha con `predict`.

In [ ]:
# Construya la clase desde Salary y conserve la partición de 200/63 filas.
# Adapte su función de cortes al Gini ponderado y ajuste DecisionTreeClassifier.
# Reporte la información de las hojas y las predicciones de los jugadores nuevos.
# Localice Alto en classes_ antes de seleccionar una columna de predict_proba.

**Interpretación y respuesta:**

_Escriba aquí su análisis y justificación._

## 8. Matriz de confusión y comparación de clasificadores

Con **Alto** como clase positiva, obtenga la matriz de confusión de prueba, la tasa de clasificación correcta (**TCC**), la tasa de error, la sensibilidad y la especificidad del árbol del punto 7, de un árbol de profundidad máxima **1** y de una regresión logística con **Years** y **Hits** (umbral **0.5**).

**Discuta** qué revela la matriz de confusión que la TCC por sí sola no muestra.

In [ ]:
# Ajuste los tres clasificadores únicamente con el entrenamiento.
# Use regresión logística sin regularización dentro del pipeline indicado.
# Construya las matrices con labels=["Bajo", "Alto"] y calcule las métricas.
# Compare los resultados en una tabla.

**Interpretación y respuesta:**

_Escriba aquí su análisis y justificación._